# 3D Mesh Quality Control - The Final Solution (V5)

This is the ultimate, brute-force notebook designed to surpass the 15.0 score mark. We are dropping all experimental architectures and losses, and using the absolute most proven Kaggle pipeline:
1. **Backbone**: `tf_efficientnetv2_s.in21k_ft_in1k` (State-of-the-art, pre-trained on 14 million images, extremely robust).
2. **Loss**: Reverted back to the successful `pos_weight` from V2, but perfectly calibrated to prevent explosions.
3. **Resolution**: Massive `512x768` for microscopic defect detection.
4. **Deterministic Quality Rule**: A strict post-processing rule that guarantees perfect `quality` correlation with the 10 defect classes. (This was the fatal flaw that dragged down the previous scores!)

In [ ]:
!pip install timm scikit-learn pandas numpy Pillow tqdm

In [ ]:
import os
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
import timm
from sklearn.metrics import f1_score
from tqdm.auto import tqdm

def seed_everything(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True

seed_everything(42)

In [ ]:
class Config:
    TRAIN_CSV = 'data/train.csv'
    TEST_CSV = 'data/test.csv'
    TRAIN_IMG_DIR = 'data/train'
    TEST_IMG_DIR = 'data/test'
    
    MODEL_NAME = 'tf_efficientnetv2_s.in21k_ft_in1k' 
    IMG_SIZE = (512, 768)
    BATCH_SIZE = 8
    EPOCHS = 10
    LR = 3e-4
    DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    DEFECTS = ['abstract','artifacts','intersection','lowpoly','noisy','open','partial','scale','set','simple']
    TARGETS = DEFECTS + ['quality']

In [ ]:
class MeshDataset(Dataset):
    def __init__(self, df, img_dir, transforms=None, is_test=False):
        self.df = df
        self.img_dir = img_dir
        self.transforms = transforms
        self.is_test = is_test

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        item_id = row['item_id']
        
        img_path = os.path.join(self.img_dir, f"{item_id}.png")
        try:
            image = Image.open(img_path).convert('RGB')
        except:
            image = Image.new('RGB', (1536, 1024))

        if self.transforms:
            image = self.transforms(image)

        if self.is_test:
            return image, item_id
        else:
            labels = row[Config.TARGETS].values.astype(np.float32)
            return image, torch.tensor(labels)

train_transform = transforms.Compose([
    transforms.Resize(Config.IMG_SIZE),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.1, contrast=0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

val_transform = transforms.Compose([
    transforms.Resize(Config.IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

In [ ]:
# Load data
train_df = pd.read_csv(Config.TRAIN_CSV)

# SPLIT 80/20
train_df_shuffled = train_df.sample(frac=1, random_state=42).reset_index(drop=True)
split_idx = int(0.8 * len(train_df_shuffled))
train_data = train_df_shuffled.iloc[:split_idx]
val_data = train_df_shuffled.iloc[split_idx:]

train_dataset = MeshDataset(train_data, Config.TRAIN_IMG_DIR, train_transform)
val_dataset = MeshDataset(val_data, Config.TRAIN_IMG_DIR, val_transform)

train_loader = DataLoader(train_dataset, batch_size=Config.BATCH_SIZE, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=Config.BATCH_SIZE, shuffle=False, num_workers=2)

# REVERTING TO POS_WEIGHT (The proven V2 secret weapon)
pos_counts = train_df[Config.TARGETS].sum()
neg_counts = len(train_df) - pos_counts
pos_weights = (neg_counts / pos_counts).values
# Cap at 25 to give massive boost to rare classes without exploding loss
pos_weights = np.clip(pos_weights, 1.0, 25.0)
pos_weights_tensor = torch.tensor(pos_weights, dtype=torch.float32).to(Config.DEVICE)
print("Using Calibrated Pos Weights:", np.round(pos_weights, 2))

In [ ]:
class MeshQualityModel(nn.Module):
    def __init__(self, model_name=Config.MODEL_NAME, pretrained=True):
        super().__init__()
        self.backbone = timm.create_model(model_name, pretrained=pretrained, num_classes=len(Config.TARGETS))

    def forward(self, x):
        return self.backbone(x)

model = MeshQualityModel().to(Config.DEVICE)
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weights_tensor)
optimizer = optim.AdamW(model.parameters(), lr=Config.LR, weight_decay=1e-3)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=Config.EPOCHS)

In [ ]:
def calculate_competition_metric(y_true, y_pred_probs, thresholds):
    y_pred = (y_pred_probs > thresholds).astype(int)
    f1_quality = f1_score(y_true[:, 10], y_pred[:, 10], zero_division=0)
    f1_defects = f1_score(y_true[:, :10], y_pred[:, :10], average='weighted', zero_division=0)
    return 10 * f1_quality + 10 * f1_defects

def optimize_thresholds(y_true, y_pred_probs):
    best_thresholds = np.zeros(len(Config.TARGETS))
    for i in range(len(Config.TARGETS)):
        best_thresh = 0.5
        best_f1 = 0
        for thresh in np.arange(0.1, 0.9, 0.05):
            preds = (y_pred_probs[:, i] > thresh).astype(int)
            score = f1_score(y_true[:, i], preds, zero_division=0)
            if score > best_f1:
                best_f1 = score
                best_thresh = thresh
        best_thresholds[i] = best_thresh
    return best_thresholds

best_metric = 0
best_val_thresholds = np.full(len(Config.TARGETS), 0.5)

for epoch in range(Config.EPOCHS):
    model.train()
    train_loss = 0
    for images, labels in tqdm(train_loader, desc=f"Epoch {epoch+1}/{Config.EPOCHS} [Train]"):
        images, labels = images.to(Config.DEVICE), labels.to(Config.DEVICE)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        # Gradient Clipping to prevent explosions
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        train_loss += loss.item()

    model.eval()
    val_loss = 0
    all_probs = []
    all_labels = []
    with torch.no_grad():
        for images, labels in tqdm(val_loader, desc=f"Epoch {epoch+1}/{Config.EPOCHS} [Val]"):
            images, labels = images.to(Config.DEVICE), labels.to(Config.DEVICE)
            outputs = model(images)
            loss = criterion(outputs, labels)
            val_loss += loss.item()
            probs = torch.sigmoid(outputs).cpu().numpy()
            all_probs.append(probs)
            all_labels.append(labels.cpu().numpy())

    scheduler.step()

    if all_probs:
        all_probs = np.vstack(all_probs)
        all_labels = np.vstack(all_labels)
        
        current_thresholds = optimize_thresholds(all_labels, all_probs)
        val_metric = calculate_competition_metric(all_labels, all_probs, current_thresholds)
    else:
        val_metric = 0
        current_thresholds = np.full(len(Config.TARGETS), 0.5)

    print(f"Epoch {epoch+1} - Train Loss: {train_loss/len(train_loader):.4f} - Val Loss: {val_loss/max(1, len(val_loader)):.4f} - Val Comp Metric: {val_metric:.4f}/20.0")

    if val_metric > best_metric:
        best_metric = val_metric
        best_val_thresholds = current_thresholds
        torch.save(model.state_dict(), 'outputs/best_model_final.pth')
        np.save('outputs/best_thresholds_final.npy', best_val_thresholds)
        print("Saved new best model (FINAL)!")

## 2. Inference & THE FATAL FLAW FIX

In [ ]:
try:
    model.load_state_dict(torch.load('outputs/best_model_final.pth'))
    best_val_thresholds = np.load('outputs/best_thresholds_final.npy')
    print("Loaded best model and thresholds:", np.round(best_val_thresholds, 2))
    model.eval()
except:
    print("Using current model weights and thresholds.")

try:
    test_df = pd.read_csv(Config.TEST_CSV)
    test_dataset = MeshDataset(test_df, Config.TEST_IMG_DIR, val_transform, is_test=True)
    test_loader = DataLoader(test_dataset, batch_size=Config.BATCH_SIZE, shuffle=False, num_workers=2)

    test_preds = []
    item_ids = []
    with torch.no_grad():
        for images, ids in tqdm(test_loader, desc="Predicting Test (TTA)"):
            images = images.to(Config.DEVICE)
            
            outputs_normal = torch.sigmoid(model(images))
            images_flipped = torch.flip(images, dims=[-1])
            outputs_flipped = torch.sigmoid(model(images_flipped))
            
            avg_probs = ((outputs_normal + outputs_flipped) / 2.0).cpu().numpy()
            preds = (avg_probs > best_val_thresholds).astype(int)
            
            test_preds.extend(preds)
            item_ids.extend(ids)

    sub_df = pd.DataFrame(test_preds, columns=Config.TARGETS)
    sub_df.insert(0, 'item_id', item_ids)

    # ==================================================================
    # THE FATAL FLAW FIX: STRICT DETERMINISTIC QUALITY RULE
    # If there are NO defects, Quality MUST BE 1. 
    # If there is ANY defect, Quality MUST BE 0.
    # We overwrite whatever the model predicted for Quality to ensure perfect points!
    # ==================================================================
    defect_sum = sub_df[Config.DEFECTS].sum(axis=1)
    sub_df['quality'] = (defect_sum == 0).astype(int)

    cols = ['item_id'] + Config.DEFECTS + ['quality']
    sub_df = sub_df[cols]
    sub_df.to_csv('outputs/submission_final.csv', index=False)
    print("Submission saved to submission_final.csv")
except Exception as e:
    print("Inference error:", e)